# Risk Analysis & P&L Visualization

Computes worst-case simultaneous unrealized loss and visualizes P&L over time for sizing capital in `THESIS.md`.

**Methodology**: For each hour in the backtest, mark every open position to market using actual hourly prices, sum their unrealized P&L, and track the worst moment.

**Supports two data sources:**
- `api` — candles from HL API (~7 months, data/candles, data/reports)
- `reservoir` — candles from Hydromancer Reservoir (~12 months, data/candles_reservoir, data/reports_reservoir_hourly)

Change `DATA_SOURCE` below to switch.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
DATA_SOURCE = 'api'  # 'api' or 'reservoir'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NOTIONAL = 50000
COINS = ['BTC', 'ETH', 'SOL', 'AVAX', 'DOGE', 'LINK']

if DATA_SOURCE == 'api':
    REPORTS_DIR = Path('../data/reports')
    CANDLES_DIR = Path('../data/candles')
elif DATA_SOURCE == 'reservoir':
    REPORTS_DIR = Path('../data/reports_reservoir_hourly')
    CANDLES_DIR = Path('../data/candles_reservoir')
else:
    raise ValueError(f'Unknown DATA_SOURCE: {DATA_SOURCE}')

print(f'Using {DATA_SOURCE} data')
print(f'  Reports: {REPORTS_DIR}')
print(f'  Candles: {CANDLES_DIR}')

## Load data

In [ ]:
# Load backtest trades
bt = pd.read_csv(REPORTS_DIR / 'backtest_trades.csv')
bt['entry_dt'] = pd.to_datetime(bt['entry_ts'], unit='ms', utc=True)
bt['exit_dt'] = pd.to_datetime(bt['exit_ts'], unit='ms', utc=True)

# Load filtered trades (correlation filter applied)
bt_filt = pd.read_csv(REPORTS_DIR / 'backtest_trades_filtered.csv')
bt_filt['entry_dt'] = pd.to_datetime(bt_filt['entry_ts'], unit='ms', utc=True)
bt_filt['exit_dt'] = pd.to_datetime(bt_filt['exit_ts'], unit='ms', utc=True)

# Load hourly candles for all coins
prices = {}
for coin in COINS:
    path = CANDLES_DIR / f'{coin}_1h.csv'
    if not path.exists():
        print(f'  Skipping {coin}: {path} not found')
        continue
    df = pd.read_csv(path)
    df['dt'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.set_index('dt').sort_index()
    df = df[~df.index.duplicated(keep='first')]
    prices[coin] = df['close']

prices = pd.DataFrame(prices).ffill().dropna()
print(f'Loaded:')
print(f'  {len(bt)} unfiltered trades')
print(f'  {len(bt_filt)} filtered trades')
print(f'  {len(prices)} hourly price bars')
print(f'  Date range: {prices.index[0]} to {prices.index[-1]}')

## P&L over time — cumulative equity curve

In [ ]:
# Load daily equity from both unfiltered and filtered backtests
eq = pd.read_csv(REPORTS_DIR / 'daily_equity.csv')
eq['date'] = pd.to_datetime(eq['date'])

eq_filt = pd.read_csv(REPORTS_DIR / 'daily_equity_filtered.csv')
eq_filt['date'] = pd.to_datetime(eq_filt['date'])

# Plot cumulative P&L for both
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: Cumulative equity curve
ax1 = axes[0]
ax1.plot(eq['date'], eq['cumulative_pnl'], label='Unfiltered', color='tab:orange', linewidth=1.5, alpha=0.7)
ax1.plot(eq_filt['date'], eq_filt['cumulative_pnl'], label='With correlation filter', color='tab:blue', linewidth=2)
ax1.axhline(0, color='black', linewidth=0.5, alpha=0.3)
ax1.set_ylabel('Cumulative P&L ($)')
ax1.set_title(f'Cumulative P&L Over Time ({DATA_SOURCE} data)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Bottom: Daily P&L bars
ax2 = axes[1]
colors = ['tab:green' if v >= 0 else 'tab:red' for v in eq_filt['daily_pnl']]
ax2.bar(eq_filt['date'], eq_filt['daily_pnl'], color=colors, alpha=0.7, label='Daily P&L (filtered)')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_ylabel('Daily P&L ($)')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

# Summary stats
print('\n=== Unfiltered ===')
print(f'  Final cumulative P&L: ${eq["cumulative_pnl"].iloc[-1]:+,.0f}')
print(f'  Best day:  ${eq["daily_pnl"].max():+,.0f} ({eq.loc[eq["daily_pnl"].idxmax(), "date"].date()})')
print(f'  Worst day: ${eq["daily_pnl"].min():+,.0f} ({eq.loc[eq["daily_pnl"].idxmin(), "date"].date()})')
print(f'  Avg daily: ${eq["daily_pnl"].mean():+,.0f}')
print(f'  Positive days: {(eq["daily_pnl"]>0).sum()}/{len(eq)} ({(eq["daily_pnl"]>0).mean()*100:.0f}%)')

print('\n=== With correlation filter ===')
print(f'  Final cumulative P&L: ${eq_filt["cumulative_pnl"].iloc[-1]:+,.0f}')
print(f'  Best day:  ${eq_filt["daily_pnl"].max():+,.0f} ({eq_filt.loc[eq_filt["daily_pnl"].idxmax(), "date"].date()})')
print(f'  Worst day: ${eq_filt["daily_pnl"].min():+,.0f} ({eq_filt.loc[eq_filt["daily_pnl"].idxmin(), "date"].date()})')
print(f'  Avg daily: ${eq_filt["daily_pnl"].mean():+,.0f}')
print(f'  Positive days: {(eq_filt["daily_pnl"]>0).sum()}/{len(eq_filt)} ({(eq_filt["daily_pnl"]>0).mean()*100:.0f}%)')

## Drawdown visualization

In [ ]:
# Compute rolling peak and drawdown
eq_filt['running_peak'] = eq_filt['cumulative_pnl'].cummax()
eq_filt['drawdown'] = eq_filt['cumulative_pnl'] - eq_filt['running_peak']

eq['running_peak'] = eq['cumulative_pnl'].cummax()
eq['drawdown'] = eq['cumulative_pnl'] - eq['running_peak']

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(eq['date'], eq['drawdown'], 0, color='tab:orange', alpha=0.4, label='Unfiltered')
ax.fill_between(eq_filt['date'], eq_filt['drawdown'], 0, color='tab:blue', alpha=0.6, label='With correlation filter')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Drawdown ($)')
ax.set_xlabel('Date')
ax.set_title(f'Peak-to-Trough Drawdown ({DATA_SOURCE} data)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

print(f'Unfiltered max drawdown: ${eq["drawdown"].min():,.0f}')
print(f'Filtered max drawdown:   ${eq_filt["drawdown"].min():,.0f}')

## Compute simultaneous unrealized P&L hour by hour

In [ ]:
def position_unrealized(trade, current_a, current_b):
    """Unrealized P&L for a single position at current prices."""
    ea, eb = trade['entry_price_a'], trade['entry_price_b']
    if trade['direction'] == 'long_ratio':
        return NOTIONAL * (current_a - ea) / ea + NOTIONAL * (eb - current_b) / eb
    else:
        return NOTIONAL * (ea - current_a) / ea + NOTIONAL * (current_b - eb) / eb

def compute_concurrent_unrealized(trades_df, prices_df):
    """For each hour, compute actual simultaneous unrealized P&L across open positions."""
    rows = []
    for hour in prices_df.index:
        active = trades_df[(trades_df['entry_dt'] <= hour) & (trades_df['exit_dt'] > hour)]
        if len(active) == 0:
            rows.append({'hour': hour, 'n_positions': 0, 'unrealized': 0.0})
            continue

        total = 0.0
        for _, t in active.iterrows():
            coin_a, coin_b = t['pair'].split('/')
            if coin_a not in prices_df.columns or coin_b not in prices_df.columns:
                continue
            total += position_unrealized(t, prices_df.loc[hour, coin_a], prices_df.loc[hour, coin_b])

        rows.append({'hour': hour, 'n_positions': len(active), 'unrealized': total})
    return pd.DataFrame(rows)

print('Computing concurrent unrealized for unfiltered backtest...')
conc = compute_concurrent_unrealized(bt, prices)
print(f'  Done: {len(conc)} hours')

print('Computing concurrent unrealized for filtered backtest...')
conc_filt = compute_concurrent_unrealized(bt_filt, prices)
print(f'  Done: {len(conc_filt)} hours')

## Simultaneous unrealized over time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(conc['hour'], conc['unrealized'], label='Unfiltered', color='tab:orange', linewidth=0.8, alpha=0.7)
ax.plot(conc_filt['hour'], conc_filt['unrealized'], label='With correlation filter', color='tab:blue', linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(-19657, color='red', linestyle='--', linewidth=0.8, label=f'API worst ($-19,657)')
ax.set_ylabel('Simultaneous Unrealized P&L ($)')
ax.set_xlabel('Date')
ax.set_title(f'Simultaneous Unrealized P&L Across All Open Positions ({DATA_SOURCE} data)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## Distribution summary

In [ ]:
for label, c in [('Unfiltered', conc), ('Filtered', conc_filt)]:
    wp = c[c['n_positions'] > 0]
    print(f'=== {label} ({DATA_SOURCE}) ===')
    print(f'  Worst:    ${wp["unrealized"].min():+,.0f}')
    print(f'  5th %ile: ${wp["unrealized"].quantile(0.05):+,.0f}')
    print(f'  Median:   ${wp["unrealized"].median():+,.0f}')
    print(f'  95th %ile:${wp["unrealized"].quantile(0.95):+,.0f}')
    print(f'  Best:     ${wp["unrealized"].max():+,.0f}')
    print()
    print(f'  Time in concurrent positions:')
    for n in sorted(c['n_positions'].unique()):
        pct = (c['n_positions'] == n).sum() / len(c) * 100
        print(f'    {n} positions: {pct:.1f}% of hours')
    print()

## Worst moment — full breakdown (uses filtered trades)

In [ ]:
# Use filtered backtest for the worst-moment analysis (this is what we'd actually trade)
worst_idx = conc_filt['unrealized'].idxmin()
worst_hour = conc_filt.loc[worst_idx, 'hour']
worst_total = conc_filt.loc[worst_idx, 'unrealized']

print(f'Worst moment (filtered): {worst_hour}')
print(f'Total simultaneous unrealized: ${worst_total:+,.0f}')
print()

print('Spot prices at that moment:')
for coin in COINS:
    if coin in prices.columns:
        print(f'  {coin:>6}: ${prices.loc[worst_hour, coin]:,.4f}')
print()

active = bt_filt[(bt_filt['entry_dt'] <= worst_hour) & (bt_filt['exit_dt'] > worst_hour)].copy()
active = active.sort_values('entry_dt')

for _, t in active.iterrows():
    coin_a, coin_b = t['pair'].split('/')
    cur_a = prices.loc[worst_hour, coin_a]
    cur_b = prices.loc[worst_hour, coin_b]
    ea, eb = t['entry_price_a'], t['entry_price_b']

    if t['direction'] == 'long_ratio':
        pnl_a = NOTIONAL * (cur_a - ea) / ea
        pnl_b = NOTIONAL * (eb - cur_b) / eb
        dir_desc = f'LONG {coin_a} / SHORT {coin_b}'
    else:
        pnl_a = NOTIONAL * (ea - cur_a) / ea
        pnl_b = NOTIONAL * (cur_b - eb) / eb
        dir_desc = f'SHORT {coin_a} / LONG {coin_b}'

    total = pnl_a + pnl_b
    hours_held = (worst_hour - t['entry_dt']).total_seconds() / 3600

    print(f'  {t["pair"]} — {dir_desc}')
    print(f'    Entry: {t["entry_dt"]} at z={t["entry_z"]:+.2f}')
    print(f'    Held: {hours_held:.0f}h (full hold: {t["hours_held"]}h, exit at {t["exit_dt"]})')
    print(f'    {coin_a}: ${ea:,.4f} → ${cur_a:,.4f} ({(cur_a-ea)/ea*100:+.2f}%) → leg P&L: ${pnl_a:+,.0f}')
    print(f'    {coin_b}: ${eb:,.4f} → ${cur_b:,.4f} ({(cur_b-eb)/eb*100:+.2f}%) → leg P&L: ${pnl_b:+,.0f}')
    print(f'    Position unrealized at worst: ${total:+,.0f}')
    print(f'    Final outcome: ${t["net_pnl"]:+,.0f} (exit z={t["exit_z"]:+.2f}, {t["exit_reason"]})')
    print()

## Timeline around worst moment (±12 hours)

In [ ]:
window_start = worst_hour - pd.Timedelta(hours=12)
window_end = worst_hour + pd.Timedelta(hours=12)
window = conc_filt[(conc_filt['hour'] >= window_start) & (conc_filt['hour'] <= window_end)].copy()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(window['hour'], window['unrealized'], marker='o', color='tab:blue', linewidth=2)
ax.axvline(worst_hour, color='red', linestyle='--', alpha=0.5, label=f'Worst moment ({worst_hour.strftime("%m-%d %H:%M")})')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Simultaneous Unrealized P&L ($)')
ax.set_xlabel('Hour')
ax.set_title('Combined Unrealized P&L Around Worst Moment (±12h)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Table view
print(f'{"Hour":>20} {"N Pos":>7} {"Unrealized":>15}')
print(f'{"-"*20} {"-"*7} {"-"*15}')
for _, row in window.iterrows():
    marker = '  ← WORST' if row['hour'] == worst_hour else ''
    print(f'{row["hour"].strftime("%Y-%m-%d %H:%M"):>20} {row["n_positions"]:>7} ${row["unrealized"]:>+13,.0f}{marker}')

## Capital requirements by leverage

In [ ]:
worst_case = abs(worst_total)
max_notional = 4 * 2 * NOTIONAL  # 4 pairs x 2 legs x $50K

print(f'Max notional (4 pairs, 2 legs, ${NOTIONAL:,}/leg): ${max_notional:,}')
print(f'Worst-case simultaneous unrealized ({DATA_SOURCE}, filtered): ${worst_case:,.0f}')
print()
print(f'{"Leverage":>9} {"Margin":>10} {"Buffer":>10} {"Capital":>10} {"Cushion":>9} {"Verdict":>8}')
print('-' * 65)

for lev in [1, 2, 3, 5, 7, 10, 15, 20]:
    margin = max_notional / lev
    if lev <= 3:
        buffer = max(50000, worst_case * 2)
    elif lev <= 5:
        buffer = 40000
    else:
        buffer = 20000
    capital = margin + buffer
    cushion = buffer / worst_case
    verdict = '✅' if cushion >= 1.5 else '⚠️' if cushion >= 1.0 else '❌'
    print(f'{lev:>7}x  ${margin:>8,.0f} ${buffer:>8,.0f} ${capital:>8,.0f} {cushion:>7.1f}x {verdict:>8}')

print()
print('Cushion = buffer / worst-case simultaneous unrealized')

## Worked example: $120K at 5x during the worst moment

In [ ]:
capital = 120000
leverage = 5
margin = max_notional / leverage
maint_margin = margin / 2  # rough approximation

equity_at_worst = capital + worst_total  # worst_total is negative
headroom = equity_at_worst - maint_margin
loss_to_liquidation = capital - maint_margin

print(f'Setup:')
print(f'  Capital:              ${capital:,}')
print(f'  Leverage:             {leverage}x')
print(f'  Total notional:       ${max_notional:,}')
print(f'  Initial margin:       ${margin:,.0f}')
print(f'  Free buffer:          ${capital - margin:,.0f}')
print(f'  Maintenance margin:   ~${maint_margin:,.0f}')
print()
print(f'At worst moment (-${worst_case:,.0f} unrealized):')
print(f'  Account equity:       ${equity_at_worst:,.0f}')
print(f'  Distance to liquidation: ${headroom:,.0f}')
print(f'  % of headroom used:   {(worst_case / loss_to_liquidation) * 100:.0f}%')
print()
print(f'To actually liquidate:')
print(f'  Required loss:        ${loss_to_liquidation:,.0f}')
print(f'  Multiple of worst:    {loss_to_liquidation / worst_case:.1f}x'),

## Export results

In [ ]:
# Save the concurrent unrealized time series
out_path = REPORTS_DIR / 'concurrent_unrealized.csv'
conc_filt.to_csv(out_path, index=False)
print(f'Saved filtered concurrent unrealized to {out_path}')

out_path_uf = REPORTS_DIR / 'concurrent_unrealized_unfiltered.csv'
conc.to_csv(out_path_uf, index=False)
print(f'Saved unfiltered concurrent unrealized to {out_path_uf}')